<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/20260824/notebooks/protein_structure_localization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 서열 → 구조 + 세포내 위치 예측 (언어모델·구조모델 응용)

**AI 신약개발 · 단백질 표현법(2026-08-24) · 실습 NB2**

아미노산 **서열 하나**를 주면:
1. **위치 예측(localization)** — 단백질 언어모델 **ESM-2** 임베딩 → 세포내 위치 분류기
2. **구조 예측(structure)** — **ESMFold API**(무설치·즉시, **Colab 권장**)로 3D 구조 + 신뢰도
   (더 정확한 **Boltz-2**(AF3급·친화도)는 §4에 선택 제공 — **로컬 GPU 권장**)

> ⚠️ **무-날조**: localization 학습 라벨은 UniProt 실주석, 정확도는 실측(교차검증), 구조는 모델 예측값(신뢰도 pLDDT/pTM 함께 표시).
> 🖥️ **Colab에서는 §1~§3만으로 완결**됩니다(설치·GPU 불필요; localization은 가벼운 ESM-2). §4 Boltz-2는 설치가 무거워 로컬/전용 GPU에서 권장.

## 0. 설치 & 환경

In [ ]:
# 위치 예측용: biopython (Colab엔 torch/transformers/sklearn 사전설치)
!pip install -q biopython py3Dmol
# 한글 폰트(그래프)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true
import matplotlib as mpl, matplotlib.font_manager as fm
_kf=[f for f in fm.findSystemFonts() if "Nanum" in f]
for _f in _kf: fm.fontManager.addfont(_f)
if _kf: mpl.rcParams["font.family"]="NanumGothic"
mpl.rcParams["axes.unicode_minus"]=False
import platform; print("python", platform.python_version(), "| 한글폰트:", "OK" if _kf else "기본")


## 1. 입력 서열

기본값은 실제 단백질입니다. **`QUERY_SEQ` 를 바꿔** 원하는 서열로 실습하세요.
(예시: 사람 리소자임 C, UniProt P61626 — 분비(Secreted) 효소, 148 aa)

In [ ]:
QUERY_NAME = "LYSC_HUMAN (P61626)"   # 분비 효소 — 아래 4개 위치 클래스 중 Secreted에 해당
QUERY_SEQ = ("MKALIVLGLVLLSVTVQGKVFERCELARTLKRLGMDGYRGISLANWMCLAKWESGYNTRATNYNAGDRST"
             "DYGIFQINSRYWCNDGKTPGAVNACHLSCSALLQDNIADAVACAKRVVRDPQGIRAWVAWRNRCQNRDVRQYVQGCGV")
print(QUERY_NAME, "| 길이", len(QUERY_SEQ))


## 2. 위치 예측 (localization) — ESM-2 표현 → 분류기

UniProt(사람·reviewed)에서 위치가 명확한 단백질을 위치별로 실제 수집(다중위치 제외) →
**ESM-2 임베딩**을 입력으로 로지스틱 회귀를 학습하고, 입력 서열의 위치를 예측합니다.

In [ ]:
import numpy as np, pandas as pd, urllib.parse, urllib.request, random, torch
from transformers import AutoTokenizer, AutoModel
random.seed(42)

AA="ACDEFGHIKLMNPQRSTVWY"; AA_SET=set(AA)
def clean(s): return "".join(c for c in s.upper() if c in AA_SET)

ESM_NAME="facebook/esm2_t6_8M_UR50D"
device="cuda" if torch.cuda.is_available() else "cpu"
tok=AutoTokenizer.from_pretrained(ESM_NAME); esm=AutoModel.from_pretrained(ESM_NAME).to(device).eval()

@torch.no_grad()
def esm_embed(seq):
    enc=tok(clean(seq),return_tensors="pt",truncation=True,max_length=1022).to(device)
    h=esm(**enc).last_hidden_state[0]
    return h[1:-1].mean(0).cpu().numpy()
print("ESM-2 로드:", ESM_NAME, "| device:", device)


In [ ]:
# 위치 4종 (UniProt keyword) 실라벨 수집 — 다중위치 제외, 클래스별 균형
LOC_KW={"Nucleus":"KW-0539","Secreted":"KW-0964","Mitochondrion":"KW-0496","CellMembrane":"KW-1003"}
PER_FETCH, N_PER = 80, 40
def uniprot_by_keyword(code,size):
    q=f"(organism_id:9606) AND (reviewed:true) AND (keyword:{code}) AND (length:[80 TO 500])"
    url="https://rest.uniprot.org/uniprotkb/search?"+urllib.parse.urlencode(
        {"query":q,"fields":"accession,sequence","format":"tsv","size":size})
    with urllib.request.urlopen(url,timeout=60) as r:
        rows=r.read().decode().splitlines()
    return [tuple(l.split("\t")[:2]) for l in rows[1:] if l]

seqmap,classes={},{}
for loc,code in LOC_KW.items():
    for acc,seq in uniprot_by_keyword(code,PER_FETCH):
        seqmap[acc]=seq; classes.setdefault(acc,set()).add(loc)
single={a:next(iter(c)) for a,c in classes.items() if len(c)==1}
by={}
for a,l in single.items(): by.setdefault(l,[]).append(a)
sel=[]
for l,accs in by.items():
    accs=sorted(accs); random.shuffle(accs)
    sel+=[(a,seqmap[a],l) for a in accs[:N_PER]]
loc_df=pd.DataFrame(sel,columns=["acc","seq","loc"])
print("학습 데이터:", len(loc_df), loc_df["loc"].value_counts().to_dict())


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

Xl=np.vstack([esm_embed(s) for s in loc_df["seq"]]); yl=loc_df["loc"].values
def clf(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv=StratifiedKFold(5,shuffle=True,random_state=42)
acc=cross_val_score(clf(),Xl,yl,cv=cv,scoring="accuracy")
print(f"ESM-2 표현 5-fold 정확도(실측) = {acc.mean():.3f} ± {acc.std():.3f}  (무작위 {1/loc_df['loc'].nunique():.2f})")

model=clf().fit(Xl,yl)
prob=model.predict_proba(esm_embed(QUERY_SEQ).reshape(1,-1))[0]
print(f"\n[{QUERY_NAME}] 위치 예측:")
for i in np.argsort(prob)[::-1]:
    print(f"  {model.classes_[i]:14} {prob[i]:.3f}")
print("\n주의: 학습된 4개 위치 중에서만 고름 →", list(model.classes_),
      "\n     (세포질 등 그 외 위치는 가장 가까운 것으로 오배정될 수 있음)")


### 2-A. 위치 분류 성능 & 임베딩 시각화 (실측·무-날조)

아래 그림은 모두 **실제 계산값**입니다.
- 혼동행렬/지표: `cross_val_predict` 로 얻은 **out-of-fold 예측**(각 표본이 학습에 안 쓰인 fold에서 예측됨)
- 임베딩 투영: 160개 단백질의 **ESM-2 임베딩** 실벡터
- 질의 막대: 입력 서열의 실제 `predict_proba` 확률

In [ ]:
# === [시각화 2-1] localization 혼동행렬 + per-class Precision/Recall/F1 ===
# 무-날조: cross_val_predict 로 얻은 out-of-fold 예측(clf/Xl/yl/cv 재사용)
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib as mpl
_fam = mpl.rcParams["font.family"]           # 기존 한글 폰트 보존
sns.set_theme(style="whitegrid", context="notebook")
mpl.rcParams["font.family"] = _fam           # set_theme가 덮어쓴 폰트 복원
mpl.rcParams["axes.unicode_minus"] = False
labels = sorted(np.unique(yl))
y_oof = cross_val_predict(clf(), Xl, yl, cv=cv)          # 실제 out-of-fold 예측
cm = confusion_matrix(yl, y_oof, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # 행(정답) 기준 정규화

rep = classification_report(yl, y_oof, labels=labels, output_dict=True, zero_division=0)
prf = np.array([[rep[l]["precision"], rep[l]["recall"], rep[l]["f1-score"]] for l in labels])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), dpi=130)

# (A) 정규화 혼동행렬 (annot %) + 표본 수
ax = axes[0]
sns.heatmap(cm_norm * 100, annot=True, fmt=".1f", cmap="Blues", vmin=0, vmax=100,
            cbar_kws={"label": "행 정규화 비율 (%)"}, xticklabels=labels, yticklabels=labels,
            ax=ax, annot_kws={"fontsize": 11}, linewidths=.5, linecolor="white")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j + 0.5, i + 0.74, f"n={cm[i, j]}", ha="center", va="center",
                fontsize=8, color="0.35")
ax.set_title(f"혼동행렬 (out-of-fold, 행 정규화)\n전체 정확도 = {np.trace(cm) / cm.sum():.3f}",
             fontsize=12)
ax.set_xlabel("예측 위치"); ax.set_ylabel("실제 위치 (UniProt 주석)")

# (B) 클래스별 Precision / Recall / F1 (colorblind 팔레트)
ax = axes[1]
x = np.arange(len(labels)); w = 0.26
pal = sns.color_palette("colorblind", 3)
for k, (metric, color) in enumerate(zip(["Precision", "Recall", "F1"], pal)):
    bars = ax.bar(x + (k - 1) * w, prf[:, k], width=w, label=metric,
                  color=color, edgecolor="white")
    for bp in bars:
        ax.text(bp.get_x() + bp.get_width() / 2, bp.get_height() + 0.012,
                f"{bp.get_height():.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
ax.set_ylim(0, 1.10); ax.set_ylabel("점수")
ax.set_title("클래스별 Precision / Recall / F1 (out-of-fold)", fontsize=12)
ax.axhline(rep["macro avg"]["f1-score"], ls="--", c="0.4", lw=1.2,
           label=f"macro-F1 = {rep['macro avg']['f1-score']:.2f}")
ax.legend(loc="lower right", frameon=True, fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# === [시각화 2-2] ESM-2 임베딩 2D 투영 (위치별 색상) ===
# Colab: UMAP 미설치 시 설치. 실패하면 PCA로 자동 대체.
!pip install -q umap-learn
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Xl_std = StandardScaler().fit_transform(Xl)          # 160 x d 실임베딩
try:
    import umap
    emb2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(Xl_std)
    method = "UMAP"
except Exception as e:
    emb2d = PCA(n_components=2, random_state=42).fit_transform(Xl_std)
    method = "PCA"
    print("UMAP 사용 불가 → PCA 대체:", repr(e))

labels = sorted(np.unique(yl))
pal = dict(zip(labels, sns.color_palette("colorblind", len(labels))))
fig, ax = plt.subplots(figsize=(8.6, 7), dpi=130)
for l in labels:
    m = (yl == l)
    ax.scatter(emb2d[m, 0], emb2d[m, 1], s=62, alpha=.85, color=pal[l],
               edgecolor="white", linewidth=.6, label=f"{l} (n={int(m.sum())})")
ax.set_title(f"ESM-2 임베딩 {method} 2D 투영 — 세포내 위치별 군집\n"
             f"(총 {len(yl)}개 단백질 · {ESM_NAME.split('/')[-1]})", fontsize=12)
ax.set_xlabel(f"{method}-1"); ax.set_ylabel(f"{method}-2")
ax.legend(title="위치 (UniProt)", frameon=True, loc="best", fontsize=9)
sns.despine(ax=ax); plt.tight_layout(); plt.show()

In [ ]:
# === [시각화 2-3] 질의 서열 위치 예측 확률 (수평 막대, 정렬) ===
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
order = np.argsort(prob)                       # 오름차순 → 큰 값이 위로
cls = model.classes_[order]; p = prob[order]
top_cls = model.classes_[int(np.argmax(prob))]
colors = ["#DD8452" if c == top_cls else "#4C72B0" for c in cls]   # 예측 클래스 강조(주황)

fig, ax = plt.subplots(figsize=(8.6, 4.3), dpi=130)
ax.barh(range(len(cls)), p, color=colors, edgecolor="white")
ax.set_yticks(range(len(cls))); ax.set_yticklabels(cls)
for i, v in enumerate(p):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)
ax.set_xlim(0, 1.06); ax.set_xlabel("예측 확률")
ax.set_title(f"[{QUERY_NAME}] 위치 예측 확률\n"
             f"최고 예측: {top_cls} ({float(np.max(prob)):.3f})", fontsize=12)
ax.axvline(1 / len(prob), ls="--", c="0.4", lw=1.2,
           label=f"무작위 기준 {1 / len(prob):.2f}")
ax.legend(loc="lower right", fontsize=9); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

## 3. 구조 예측 — ESMFold API (무설치·즉시 · **Colab 권장**)

Colab에서는 **ESM Atlas의 ESMFold API**로 서열→구조를 받습니다.
**설치·GPU·다운로드가 전혀 없어** 수 초 만에 3D 구조가 나옵니다(단일 서열 기준, 길이 제한 있음).

> Colab 환경(numpy 등 최신 스택)에서 무거운 모델을 새로 깔 필요가 없어, 수업용으로 가장 매끄럽습니다.

In [ ]:
import urllib.request, numpy as np, py3Dmol
def esmfold_api(seq):
    req=urllib.request.Request("https://api.esmatlas.com/foldSequence/v1/pdb/",
                               data=clean(seq).encode(), method="POST")
    with urllib.request.urlopen(req, timeout=120) as r:
        return r.read().decode()

pdb = esmfold_api(QUERY_SEQ)
# 평균 pLDDT = B-factor 열 평균 (ESMFold는 B-factor에 pLDDT 기록). API 출력이 0~1 스케일이면 ×100.
b=[float(l[60:66]) for l in pdb.splitlines() if l.startswith("ATOM") and l[12:16].strip()=="CA"]
scale01 = max(b) <= 1.5
mplddt = (np.mean(b)*100) if scale01 else np.mean(b)
print(f"ESMFold 평균 pLDDT ≈ {mplddt:.1f}  (0~100, 높을수록 신뢰)")
lo, hi = (0.5, 0.9) if scale01 else (50, 90)   # 색상 범위도 스케일에 맞춤
view=py3Dmol.view(width=640,height=480)
view.addModel(pdb,"pdb")
view.setStyle({"cartoon":{"colorscheme":{"prop":"b","gradient":"roygb","min":lo,"max":hi}}})
view.zoomTo(); view.show()


### 3-A. ESMFold 구조 신뢰도 프로파일 (실측 pLDDT)

py3Dmol 3D 리본은 pLDDT를 **roygb** 그라디언트로 칠합니다(빨강=낮음 → 파랑=높음).
아래 라인플롯은 동일한 pLDDT를 **잔기 번호 축**으로 펼쳐, 어느 구간이 불확실한지 정량적으로 보여줍니다.

In [ ]:
# === [시각화 3] ESMFold 잔기별 pLDDT 프로파일 (§3의 pdb/b/scale01 재사용) ===
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.patches import Patch

plddt = (np.array(b) * 100) if scale01 else np.array(b)   # 0~100 스케일 통일
resi = np.arange(1, len(plddt) + 1)

fig, ax = plt.subplots(figsize=(13, 4.9), dpi=130)
# pLDDT 밴드 shading (AlphaFold/ESMFold 관례 색)
bands = [(0, 50, "#FF7D45", "매우 낮음 <50"), (50, 70, "#FFDB13", "낮음 50–70"),
         (70, 90, "#65CBF3", "높음 70–90"), (90, 100, "#0053D6", "매우 높음 >90")]
for lo_, hi_, col, _lab in bands:
    ax.axhspan(lo_, hi_, color=col, alpha=.13)
ax.fill_between(resi, plddt, 0, color="#4C72B0", alpha=.12, zorder=1)
ax.plot(resi, plddt, color="#222222", lw=1.6, zorder=3)
mp = float(plddt.mean())
ax.axhline(mp, ls="--", c="crimson", lw=1.4, zorder=4)
ax.set_xlim(1, len(plddt)); ax.set_ylim(0, 100)
ax.set_xlabel("잔기 번호 (residue)"); ax.set_ylabel("pLDDT (0–100)")
ax.set_title(f"ESMFold 잔기별 신뢰도 프로파일 — {QUERY_NAME}\n"
             f"(길이 {len(plddt)} aa · 높을수록 신뢰)", fontsize=12)
handles = [Patch(color=c, alpha=.35, label=l) for _lo, _hi, c, l in bands]
handles.append(plt.Line2D([0], [0], ls="--", c="crimson", label=f"평균 pLDDT = {mp:.1f}"))
ax.legend(handles=handles, loc="lower right", ncol=2, fontsize=8, framealpha=.92)
sns.despine(ax=ax); plt.tight_layout(); plt.show()
print("참고: py3Dmol 3D 색상(roygb) = pLDDT — 빨강=낮음 → 파랑=높음 (위 밴드와 동일 의미)")

## 4. (선택·고급) 구조 예측 — Boltz-2 (AlphaFold3급 · **로컬 GPU 권장**)

**Boltz-2**(Wohlwend et al., 2025)는 서열에서 3D 구조(및 복합체·**결합친화도**)를 예측하는 오픈모델로, 정확도가 더 높습니다.

> ⚠️ **Colab 주의**: Colab의 최신 스택(numpy 2.x 등)과 boltz의 예전 의존성이 충돌해, 설치 시 여러 패키지를 재빌드하느라 **최초 5~10분** 걸립니다(설치 끝의 'dependency conflicts' 빨간 줄은 경고). **로컬 GPU 환경**에서는 설치가 순조롭고 예측이 ~8초로 빠릅니다 — Boltz-2는 로컬/전용 GPU에서 쓰길 권장합니다. 시간이 없으면 위 §3(ESMFold)만으로 충분합니다.

In [ ]:
# Boltz-2 설치 (선택). Colab에선 5~10분(1회). uv 리졸버로 앞당김(실패 시 pip).
!pip install -q uv && uv pip install --system -q boltz || pip install -q -U boltz


In [ ]:
# 입력 YAML 작성 (공식 형식: version/sequences/protein/id/sequence)
import os, glob, json
os.makedirs("boltz_in", exist_ok=True)
yaml_text = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {clean(QUERY_SEQ)}
"""
with open("boltz_in/query.yaml","w") as f: f.write(yaml_text)
print(yaml_text)


In [ ]:
# 예측 실행 (GPU). MSA는 원격 서버 사용. 최초 실행 시 가중치 다운로드.
# --no_kernels: cuequivariance 최적화 커널을 끔(환경 독립적으로 안정). 커널 설치가 되면 빼면 더 빠름.
!boltz predict boltz_in/query.yaml --use_msa_server --out_dir boltz_out --override --no_kernels

cif = sorted(glob.glob("boltz_out/**/*_model_0.cif", recursive=True))
conf = sorted(glob.glob("boltz_out/**/confidence_*_model_0.json", recursive=True))
print("구조 파일:", cif[:1])
if conf:
    c=json.load(open(conf[0]))
    print("신뢰도: confidence_score=%.3f | ptm=%.3f | complex_plddt=%.3f" %
          (c.get("confidence_score",float("nan")), c.get("ptm",float("nan")), c.get("complex_plddt",float("nan"))))


In [ ]:
# py3Dmol 시각화 (Boltz-2 출력 cif). pLDDT(B-factor)로 색칠.
import py3Dmol
if cif:
    view=py3Dmol.view(width=640,height=480)
    view.addModel(open(cif[0]).read(),"cif")
    view.setStyle({"cartoon":{"colorscheme":{"prop":"b","gradient":"roygb","min":50,"max":90}}})
    view.zoomTo(); view.show()
else:
    print("cif 없음 — 위 §3(ESMFold)를 쓰거나 Boltz 실행 로그를 확인하세요.")


In [ ]:
# === [시각화 4] (선택·§4) Boltz-2 신뢰도 요약 — 출력 파일 있을 때만 (skip 가드) ===
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, json, glob
_conf = sorted(glob.glob("boltz_out/**/confidence_*_model_0.json", recursive=True))
if _conf:
    c = json.load(open(_conf[0]))
    keys = ["confidence_score", "ptm", "iptm", "complex_plddt", "complex_iplddt"]
    vals = [(k, float(c[k])) for k in keys if isinstance(c.get(k), (int, float))]
    if vals:
        names = [k for k, _ in vals]; sc = [v for _, v in vals]
        pal = sns.color_palette("colorblind", len(sc))
        fig, ax = plt.subplots(figsize=(8.6, 4.6), dpi=130)
        bars = ax.bar(names, sc, color=pal, edgecolor="white")
        for bp, v in zip(bars, sc):
            ax.text(bp.get_x() + bp.get_width() / 2, v + 0.012, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=10)
        ax.set_ylim(0, max(1.0, max(sc) * 1.15)); ax.set_ylabel("점수 (0–1)")
        ax.set_title(f"Boltz-2 예측 신뢰도 요약 — {QUERY_NAME}", fontsize=12)
        ax.axhline(0.5, ls="--", c="0.4", lw=1.2, label="0.5 기준")
        plt.xticks(rotation=15); ax.legend(loc="lower right", fontsize=9)
        sns.despine(ax=ax); plt.tight_layout(); plt.show()
    else:
        print("confidence json에 수치 항목이 없어 skip.")
else:
    print("Boltz 출력(confidence json) 없음 — §4를 실행하지 않았거나 로컬 GPU 필요. skip.")

## 5. 정리

- **위치**: ESM-2(언어모델) 임베딩만으로도 세포내 위치를 상당히 맞힘 — "좋은 표현 → downstream 성능".
- **구조**: Boltz-2(GPU, AF3급)로 3D 구조·신뢰도, 또는 ESMFold API(무GPU)로 경량 예측.
- **신뢰도 필수 확인**: pLDDT/pTM가 낮은 영역은 불확실 — 예측 구조는 **가설**이며 실험(X-ray/Cryo-EM) 검증 전까지 결론이 아님.

**참고문헌**
- Lin et al. *Science* 379:1123 (2023) — ESM-2 / ESMFold
- Wohlwend et al. *Boltz-1/2* (2024–2025) — 오픈 구조·친화도 예측
- Jumper et al. *Nature* 596:583 (2021); Abramson et al. *Nature* (2024) — AlphaFold2/3
- Cock et al. *Bioinformatics* 25:1422 (2009) — Biopython